# EDA — corpus overview and descriptive statistics

Exploratory summaries of the annotated corpus and the cross-community corpus, reproducing the descriptive figures reported in the ERP (class balance, comments per channel, comment length, and RQ2 community composition). Runs from the included data — no collection or GPU required.

Reads `data/processed/` (labelled corpus and splits) and `outputs/rq2/` (scored corpus); writes figures to `outputs/eda/`.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
proc = PROJECT_ROOT / "data" / "processed"
figs = PROJECT_ROOT / "outputs" / "eda"
figs.mkdir(parents=True, exist_ok=True)

labelled = pd.read_csv(proc / "labelled_comments_final.csv")
label_cols = ["label_1A", "label_1B", "label_2"]
for c in label_cols:
    labelled[c] = pd.to_numeric(labelled[c], errors="coerce").astype(int)
print("Annotated corpus:", len(labelled), "comments")


## 1. Class balance of the annotated corpus

In [ ]:
n = len(labelled)
balance = pd.DataFrame({
    "positive_n": [int(labelled[c].sum()) for c in label_cols],
    "positive_pct": [round(100 * labelled[c].mean(), 1) for c in label_cols],
}, index=["1A grievance", "1B weaponised", "2 conspiratorial"])
print(balance)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(balance.index, balance["positive_pct"], color=["#3f6fb0", "#c0504d", "#4f8a5b"])
for i, v in enumerate(balance["positive_pct"]):
    ax.text(i, v + 0.5, f"{v}%", ha="center", fontsize=9)
ax.set_ylabel("Positive comments (%)")
ax.set_title(f"Class balance (n = {n})")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.savefig(figs / "class_balance.png", dpi=200); plt.show()


## 2. Comments per channel (annotated corpus)

In [ ]:
by_channel = labelled["channel_handle"].value_counts()
print(by_channel)
fig, ax = plt.subplots(figsize=(7, 4))
by_channel.sort_values().plot.barh(ax=ax, color="#6f93be")
ax.set_xlabel("Annotated comments"); ax.set_title("Comments per channel (annotated corpus)")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.savefig(figs / "comments_per_channel.png", dpi=200); plt.show()


## 3. Comment-length distribution

In [ ]:
labelled["n_words"] = labelled["text_clean"].fillna("").str.split().str.len()
print(labelled["n_words"].describe().round(1))
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(labelled["n_words"], bins=40, color="#9db4d0", edgecolor="white")
ax.set_xlabel("Words per comment"); ax.set_ylabel("Comments")
ax.set_title("Comment-length distribution (annotated corpus)")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.savefig(figs / "comment_length.png", dpi=200); plt.show()


## 4. Splits — sizes and positive counts

In [ ]:
rows = []
for name in ["train_fit", "validation", "test"]:
    d = pd.read_csv(proc / f"{name}.csv")
    for c in label_cols:
        d[c] = pd.to_numeric(d[c], errors="coerce").astype(int)
    rows.append({"split": name, "n": len(d),
                 "1A": int(d["label_1A"].sum()),
                 "1B": int(d["label_1B"].sum()),
                 "2":  int(d["label_2"].sum())})
print(pd.DataFrame(rows).to_string(index=False))


## 5. RQ2 cross-community corpus composition

In [ ]:
rq2 = pd.read_csv(PROJECT_ROOT / "outputs" / "rq2" / "rq2_FINAL_primary_12mo_SCORED.csv",
                  low_memory=False)
comp = rq2.groupby("community_group").agg(
    channels=("channel_name", "nunique"),
    videos=("video_id", "nunique"),
    comments=("comment_id", "count")).reset_index()
print(comp.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(comp["community_group"], comp["comments"], color="#3f6fb0")
ax.set_ylabel("Comments"); ax.set_title("RQ2 corpus size by community")
ax.tick_params(axis="x", rotation=20)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.savefig(figs / "rq2_community_size.png", dpi=200); plt.show()
